In [ ]:
#importing libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#dataset loading
df=pd.read_csv("ai_task_management_200k (1).csv")

In [ ]:
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
#basic dataset info
print("\nDataset Info")
df.info()

In [ ]:
print("\nMissing Values")
df.isnull().sum()

In [ ]:
print("\nStatistical Summary")
df.describe()

In [ ]:
#handling missing values
# Fill numeric columns
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

In [ ]:
# Fill categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna("Unknown")

In [ ]:
df['age_group'].value_counts()

In [ ]:
#age group distribution
%matplotlib inline

age_order = [
    'Teen',
    'Young Adult',
    'Adult',
    'Middle-Aged',
    'Senior'
]

plt.figure(figsize=(8, 5))

counts = df['age_group'].value_counts().reindex(age_order)

counts.plot(kind='bar', width=0.6)

plt.title("Age Group Distribution")
plt.xlabel("Age Group")
plt.ylabel("Number of Users")
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
#task category distribution
plt.figure(figsize=(10,5))
sns.countplot(data=df, x="category_ml", order=df["category_ml"].value_counts().index)
plt.xticks(rotation=45)
plt.title("Task Category Distribution")
plt.show()

In [ ]:
#Energy Level vs Task Category
plt.figure(figsize=(10,6))
sns.boxplot(x="category_ml", y="energy_level", data=df)
plt.xticks(rotation=45)
plt.title("Energy Level vs Task Category")
plt.show()

In [ ]:
#Task Priority Analysis
if "priority" in df.columns:
    plt.figure(figsize=(6,4))
    df['priority'].value_counts().plot(kind='pie', autopct="%1.1f%%")
    plt.title("Task Priority Distribution")
    plt.ylabel("")
    plt.show()

In [ ]:
sns.countplot(x='priority', data=df)
plt.title("Task Priority Distribution")
plt.show()

In [ ]:
#Correlation Analysis
plt.figure(figsize=(10,6))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
#Productivity Insights
df['category_ml'] = (df['status'] == 'Completed').astype(int)
plt.figure(figsize=(6, 4))
sns.barplot(x="energy_level",
            y="category_ml",
            data=df)
plt.title("Energy Level vs Task Completion")
plt.xlabel("Energy Level")
plt.ylabel("Completion Rate (0 to 1)")
plt.show()
print("\n📊 Completion Rate per Energy Level:")
result = df.groupby('energy_level')['category_ml'].mean().round(3) * 100
for energy, rate in result.items():
    print(f"  {energy:<10} : {rate:.1f}% tasks completed")

In [ ]:
df.columns

In [ ]:
df[['description', 'activity', 'parent_task']].head(10)

In [ ]:
df['text']=df['description']
df['text'].head()

In [ ]:
print(df['description'].nunique())

In [ ]:
#NLP preprocessing 
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

df['cleaned_text'] = df['text'].apply(preprocess_text)

df[['text', 'cleaned_text']].head()

In [ ]:
#TFIDF feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300)
X = tfidf.fit_transform(df['cleaned_text']).toarray()

In [ ]:
#target variable
y = df['category_ml'].astype(int)

In [ ]:
#train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Naive Bayes model
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score   # <-- ADD THIS LINE

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, y_pred_nb)*100)

In [ ]:
#random forest model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(
    n_estimators=50,     # default = 100 → reduce trees
    max_depth=10,        # limit depth (very important)
    n_jobs=-1,           # use all CPU cores (BIG speed boost)
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Model Accuracy:", accuracy_score(y_test, pred)*100)

In [ ]:
#SVM model
from sklearn.svm import LinearSVC 
svm_model = LinearSVC(max_iter=3000, C=1.0)
svm_model.fit(X_train, y_train) 
y_pred_svm = svm_model.predict(X_test)

print("Model Accuracy:", accuracy_score(y_test, y_pred_svm)*100)

In [ ]:
#evaluation(accuracy and classification report)
from sklearn.metrics import accuracy_score, classification_report

print("----- NAIVE BAYES -----")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

print("-----RANDOM FOREST-----")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

print("\n----- SVM -----")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

In [ ]:
#Model Comparison Graph
models = ['Naive Bayes', 'Random Forest','SVM']
accuracies = [
    accuracy_score(y_test, y_pred_nb),
    accuracy_score(y_test, pred),
    accuracy_score(y_test, y_pred_svm)
]

plt.bar(models, accuracies)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
#Overfitting check for naive bayes 
from sklearn.metrics import accuracy_score

train_acc_nb = accuracy_score(y_train, nb_model.predict(X_train))
test_acc_nb  = accuracy_score(y_test,  nb_model.predict(X_test))

print("----- NAIVE BAYES -----")
print(f"Train Accuracy : {train_acc_nb*100:.2f}%")
print(f"Test  Accuracy : {test_acc_nb*100:.2f}%")
print(f"Difference     : {(train_acc_nb - test_acc_nb)*100:.2f}%")

if (train_acc_nb - test_acc_nb) < 0.02:
    print("✅ No overfitting — model generalizes well!\n")
elif (train_acc_nb - test_acc_nb) < 0.05:
    print("⚠️ Slight overfitting — minor tuning needed\n")
else:
    print("🚨 Overfitting detected — needs fixing!\n")

In [ ]:
#overfitting check for random forest
train_acc_rf = accuracy_score(y_train, model.predict(X_train))
test_acc_rf  = accuracy_score(y_test,  model.predict(X_test))

print("----- RANDOM FOREST -----")
print(f"Train Accuracy : {train_acc_rf*100:.2f}%")
print(f"Test  Accuracy : {test_acc_rf*100:.2f}%")
print(f"Difference     : {(train_acc_rf - test_acc_rf)*100:.2f}%")

if (train_acc_rf - test_acc_rf) < 0.02:
    print("✅ No overfitting — model generalizes well!")
elif (train_acc_rf - test_acc_rf) < 0.05:
    print("⚠️ Slight overfitting — minor tuning needed")
else:
    print("🚨 Overfitting detected — needs fixing!")

In [ ]:
#Overfitting check for SVM
train_acc_svm = accuracy_score(y_train, svm_model.predict(X_train))
test_acc_svm  = accuracy_score(y_test,  svm_model.predict(X_test))

print("----- SVM -----")
print(f"Train Accuracy : {train_acc_svm*100:.2f}%")
print(f"Test  Accuracy : {test_acc_svm*100:.2f}%")
print(f"Difference     : {(train_acc_svm - test_acc_svm)*100:.2f}%")

if (train_acc_svm - test_acc_svm) < 0.02:
    print("✅ No overfitting — model generalizes well!")
elif (train_acc_svm - test_acc_svm) < 0.05:
    print("⚠️ Slight overfitting — minor tuning needed")
else:
    print("🚨 Overfitting detected — needs fixing!")

In [ ]:
#pkl file generation
import pickle

with open('nb_model.pkl', 'wb') as f:
    pickle.dump(nb_model, f)

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('svm_model.pkl', 'wb') as f:
    pickle.dump(svm_model, f)

with open('tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("✅ Models saved successfully!")

In [ ]:
import os
print(os.getcwd())

In [ ]:
# Prediction on new task
new_tasks = [
    "Complete project report urgently before deadline",
    "Watch movie in evening",
    "Prepare for exam tomorrow",
    "Go for a morning workout session",
    "Pay electricity bill before due date",
    "Clean the house on weekend",
    "Read book before sleeping tonight",
    "Finish client presentation by Friday"
]

print("Task Completion based on priority------>")
print("=" * 120)
print(f"{'TASK':<40} {'NB':^10} {'RF':^10} {'SVM':^10}")
print("=" * 120)

for task in new_tasks:
    new_cleaned  = [preprocess_text(task)]
    new_X        = tfidf.transform(new_cleaned).toarray()
    nb_pred      = nb_model.predict(new_X)[0]
    pred         = model.predict(new_X)[0]
    svm_pred     = svm_model.predict(new_X)[0]
    nb_label     = "✅ Done" if nb_pred  == 1 else "❌ Not Done"
    label        = "✅ Done" if pred  == 1 else "❌ Not Done"
    svm_label    = "✅ Done" if svm_pred == 1 else "❌ Not Done"
    print(f"{task:<40} {nb_label:^10} {label:^10} {svm_label:^10}")

print("=" * 120)

In [ ]:
def assign_priority(deadline):
    if deadline < 2:
        return "High"
    elif deadline < 5:
        return "Medium"
    else:
        return "Low"

In [ ]:
model = RandomForestClassifier(n_estimators=50, n_jobs=-1)
model.fit(X_train, y_train)

In [ ]:
team = {
    "A": 3,
    "B": 5,
    "C": 2
}

# assign to least loaded
assigned = min(team, key=team.get)
team[assigned] += 1

In [ ]:
import joblib
joblib.dump(model, "priority_model.pkl")

In [ ]:
pip install streamlit

In [ ]:
print(df["category"].value_counts())

In [ ]:
df["task_length"] = df["cleaned_text"].apply(len)
df["num_words"] = df["cleaned_text"].apply(lambda x: len(x.split()))

In [ ]:
X = df[[
    "urgency_score",
    "priority_score",
    "importance",
    "duration_minutes",
    "energy_level"
]]

In [ ]:
df.columns

In [ ]:
y = df["priority"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["importance"] = le.fit_transform(df["importance"])
df["energy_level"] = le.fit_transform(df["energy_level"])

In [ ]:
X = df[[
    "urgency_score",
    "priority_score",
    "importance",
    "duration_minutes",
    "energy_level"
]]

model.fit(X, y)

In [ ]:
print(X.dtypes)

In [ ]:
import joblib
joblib.dump(model, "priority_model.pkl")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# ✅ USE RAW TEXT
X = df["cleaned_text"]
y = df["category_ml"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=1000)),
    ("model", MultinomialNB())
])

# ✅ THIS WILL WORK
pipeline.fit(X_train, y_train)

In [ ]:
import joblib
joblib.dump(pipeline, "final_model.pkl")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
df["category_ml"].value_counts().plot(kind="bar", color="skyblue")
plt.title("Task Category Distribution")
plt.xlabel("Category")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
df["priority"].value_counts().plot(kind="bar", color="orange")
plt.title("Priority Distribution")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
df["task_length"] = df["cleaned_text"].apply(len)
df["num_words"] = df["cleaned_text"].apply(lambda x: len(str(x).split()))

In [ ]:
X = df[[
    "urgency_score",
    "priority_score",
    "importance",
    "duration_minutes",
    "energy_level",
    "task_length",
    "num_words"
]]

y = df["priority"]

In [ ]:
mapping = {"Low": 1, "Medium": 2, "High": 3, "Very High": 4}

df["importance"] = df["importance"].map(mapping)
df["energy_level"] = df["energy_level"].map(mapping)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=params,
    n_iter=5,
    cv=3,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

print("Best Params:", search.best_params_)

In [ ]:
y_pred_best = best_model.predict(X_test)

print("Final Accuracy:", accuracy_score(y_test, y_pred_best))

In [ ]:
!pip install xgboost


In [ ]:
df["importance"] = df["importance"].fillna(2)  # default = Medium

In [ ]:
print(df["importance"].describe())

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
X = df[[
    "urgency_score",
    "priority_score",
    "importance",
    "duration_minutes",
    "energy_level",
    "task_length",
    "num_words"
]]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
print(X.columns)

In [ ]:
print(pd.Series(y_test).value_counts())

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
def create_priority(row):
    score = row["urgency_score"] + row["importance"]
    
    if score >= 12:
        return "High"
    elif score >= 7:
        return "Medium"
    else:
        return "Low"

df["priority_new"] = df.apply(create_priority, axis=1)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(df["priority_new"])

In [ ]:
X = df[[
    "priority_score",
    "duration_minutes",
    "energy_level",
    "task_length",
    "num_words",
    "is_routine",
    "time_slot"
]]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_time = LabelEncoder()
df["time_slot"] = le_time.fit_transform(df["time_slot"])

In [ ]:
X = df[[
    "priority_score",
    "duration_minutes",
    "energy_level",
    "task_length",
    "num_words",
    "is_routine",
    "time_slot"   # now numeric ✅
]]

In [ ]:
print(X.dtypes)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(df["priority_new"])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
#model.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
import numpy as np

unique, counts = np.unique(y_test, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)   # MUST be X_test

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
df["urgency_x_duration"] = df["priority_score"] * df["duration_minutes"]
df["length_per_word"] = df["task_length"] / (df["num_words"] + 1)
df["effort_score"] = df["duration_minutes"] * df["energy_level"]

In [ ]:
X = df[[
    "priority_score",
    "duration_minutes",
    "task_length",
    "num_words",
    "energy_level",
    "urgency_x_duration",
    "length_per_word",
    "effort_score"
]]

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    random_state=42
)

In [ ]:
X = df[[
    "priority_score",
    "duration_minutes",
    "task_length",
    "num_words"
]]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df["priority_new"])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)

class_weights = dict(zip(classes, weights))

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=2,   # 🔥 try values like 2–5
    random_state=42
)

In [ ]:
df["priority_new2"] = df["priority_new"].replace({
    "Medium": "High"
})

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df["priority_new2"])

In [ ]:
y

In [ ]:
X = df[[
    "priority_score",
    "duration_minutes",
    "task_length",
    "num_words"
]]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

print("Final Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
nb_acc = 0.81
rf_acc = 0.79
svm_acc = 0.80
xgb_acc = 0.60

In [ ]:
models = {
    "Naive Bayes": nb_acc,
    "Random Forest": rf_acc,
    "SVM": svm_acc,
    "XGBoost": xgb_acc
}

In [ ]:
import matplotlib.pyplot as plt

names = list(models.keys())
values = list(models.values())

plt.figure(figsize=(8,5))
plt.bar(names, values)

plt.title("Model Accuracy Comparison")
plt.xlabel("Models")
plt.ylabel("Accuracy")

plt.show()

In [ ]:
model.fit(X, y)

import joblib
joblib.dump(model, "task_model.pkl")